In [ ]:
!pip install pykka

In [ ]:
import json
import logging
import time
import pykka

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)


# --- 1. DEFINICIÓN DE ACTORES ---
class WorkerActor(pykka.ThreadingActor):

  def __init__(self, worker_id: int):
    super().__init__()
    self.worker_id = worker_id

  def on_receive(self, message: dict) -> dict:
    payload = message.get("payload", "")

    # Simulación de fallo intencional
    if payload == "FAIL":
      logger.error(
          f"[Worker-{self.worker_id}] Error provocado por payload 'FAIL'"
      )
      raise ValueError("Error intencional en el Worker")

    result = payload.upper()
    logger.info(
        f"[Worker-{self.worker_id}] Tarea procesada con éxito: {result}"
    )
    return {"status": "success", "result": result, "worker_id": self.worker_id}


class SupervisorActor(pykka.ThreadingActor):

  def __init__(self):
    super().__init__()
    self.worker_ref = None
    self._spawn_worker()

  def _spawn_worker(self):
    self.worker_ref = WorkerActor.start(worker_id=1)
    logger.info("[Supervisor] Nuevo Worker iniciado correctamente.")

  def on_receive(self, message: dict) -> dict:
    payload = message.get("payload")
    try:
      response = self.worker_ref.ask({"payload": payload}, timeout=3.0)
      return response
    except Exception as e:
      logger.warning(
          f"[Supervisor] Excepción detectada en Worker: {e}. Reiniciando"
          " Worker..."
      )
      if self.worker_ref:
        self.worker_ref.stop()
      self._spawn_worker()
      return {
          "status": "error_recovered",
          "message": (
              "El Worker falló pero el Supervisor lo ha reiniciado exitosamente."
          ),
      }


# --- 2. INSTANCIACIÓN Y EJECUCIÓN DEL HANDLER ---
supervisor_ref = SupervisorActor.start()


def lambda_handler(event, context=None):
  try:
    body = (
        json.loads(event.get("body", "{}"))
        if isinstance(event.get("body"), str)
        else event
    )
    payload = body.get("payload", "hello world")

    response = supervisor_ref.ask({"payload": payload}, timeout=5.0)

    return {
        "statusCode": 200,
        "headers": {"Content-Type": "application/json"},
        "body": json.dumps(response),
    }
  except Exception as e:
    return {
        "statusCode": 500,
        "headers": {"Content-Type": "application/json"},
        "body": json.dumps({"status": "error", "message": str(e)}),
    }


# Prueba de ejecución directa en la celda
print(lambda_handler({"payload": "hola mundo"}))

{'statusCode': 200, 'headers': {'Content-Type': 'application/json'}, 'body': '{"status": "success", "result": "HOLA MUNDO", "worker_id": 1}'}


In [ ]:
# Verificamos que el Supervisor creó un nuevo Worker funcional
print(lambda_handler({"payload": "prueba despues del fallo"}))

{'statusCode': 200, 'headers': {'Content-Type': 'application/json'}, 'body': '{"status": "success", "result": "PRUEBA DESPUES DEL FALLO", "worker_id": 1}'}
